# Phase 2: Fraud Prioritization Proof of Concept

This notebook builds a fraud-risk ranking model for a fixed investigation capacity of **1% of transactions**. It compares an explainable Phase 1 rule with logistic regression and a nonlinear boosted-tree model, then evaluates both fraud-case capture and fraudulent-value capture.

**Methodology:** every candidate model is fit on the full labeled training dataset. The supplied test set is labeled, so it is used to compare the ranked queues and report POC performance. `is_fraud` is never used as an input feature or during fitting. Because test results guide the model choice in this shortened workflow, they are POC evidence rather than an independent future-production benchmark.

## Setup

Install the dependencies once if needed: `pip install pandas numpy scikit-learn matplotlib seaborn xgboost lightgbm`. This notebook requires scikit-learn 1.2 or newer, plus XGBoost and LightGBM.

In [ ]:
from pathlib import Path
from math import ceil
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

try:
    from xgboost import XGBClassifier
    from lightgbm import LGBMClassifier
except ImportError as exc:
    raise ImportError(
        'Install the tree-model dependencies in this kernel, then restart it: '
        'pip install xgboost lightgbm'
    ) from exc

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE = 42
INVESTIGATION_CAPACITY = 0.01

DATA_DIR = Path('/Users/yilingz/Downloads')
TRAIN_PATH = DATA_DIR / 'train_data.csv'
TEST_PATH = DATA_DIR / 'test_data.csv'
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

assert TRAIN_PATH.exists(), f'Missing training data: {TRAIN_PATH}'
assert TEST_PATH.exists(), f'Missing test data: {TEST_PATH}'

In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

TARGET = 'is_fraud'
EXCLUDED_FEATURES = ['sender_id', 'recipient_id', 'is_flagged_fraud', TARGET]

required_columns = {
    'time_step', 'transaction_type', 'transaction_amount', 'sender_balance_before',
    'sender_balance_after', 'recipient_balance_before', 'recipient_balance_after', TARGET
}
missing_columns = required_columns - set(train_raw.columns)
assert not missing_columns, f'Missing required columns: {sorted(missing_columns)}'

print(f'Training rows: {len(train_raw):,}')
print(f'Test rows:     {len(test_raw):,}')
print(f'Training fraud rate: {train_raw[TARGET].mean():.4%}')
if TARGET in test_raw.columns:
    print(f'Test fraud rate (evaluation-only): {test_raw[TARGET].mean():.4%}')

display(train_raw.head())
display(
    train_raw.groupby('transaction_type')[TARGET]
    .agg(transactions='size', fraud_cases='sum', fraud_rate='mean')
    .sort_values('fraud_rate', ascending=False)
)

## Feature engineering

The features deliberately represent transaction behavior rather than raw account identity. `is_flagged_fraud` is excluded because it is an existing simulated detection rule, not an independent behavioral signal. Balance-after fields are retained because they are supplied for this proof of concept and have been approved as available at scoring time.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create decision-time behavioral features without using labels or raw account IDs."""
    x = df.copy()
    amount = x['transaction_amount'].clip(lower=0)
    sender_before = x['sender_balance_before'].clip(lower=0)
    recipient_before = x['recipient_balance_before'].clip(lower=0)

    x['sender_balance_change'] = x['sender_balance_before'] - x['sender_balance_after']
    x['recipient_balance_change'] = x['recipient_balance_after'] - x['recipient_balance_before']
    x['sender_balance_error'] = x['sender_balance_before'] - amount - x['sender_balance_after']
    x['recipient_balance_error'] = x['recipient_balance_after'] - x['recipient_balance_before'] - amount
    x['sender_balance_emptied'] = (x['sender_balance_after'] == 0).astype('int8')
    x['amount_to_sender_balance'] = amount / (sender_before + 1.0)
    x['amount_to_recipient_balance'] = amount / (recipient_before + 1.0)
    x['recipient_is_merchant'] = x['recipient_id'].astype('string').str.startswith('M').astype('int8')
    x['hour_of_day'] = x['time_step'] % 24
    x['hour_sin'] = np.sin(2 * np.pi * x['hour_of_day'] / 24)
    x['hour_cos'] = np.cos(2 * np.pi * x['hour_of_day'] / 24)
    x['log_transaction_amount'] = np.log1p(amount)

    x = x.replace([np.inf, -np.inf], np.nan)
    return x.drop(columns=[column for column in EXCLUDED_FEATURES if column in x.columns])

X_all = engineer_features(train_raw)
y_all = train_raw[TARGET].astype(int)
X_test_all = engineer_features(test_raw)

assert list(X_all.columns) == list(X_test_all.columns), 'Train/test feature columns differ.'
print(f'Model features ({X_all.shape[1]}): {list(X_all.columns)}')

## Train/test setup

All candidate models train on every labeled training row. The labeled test data is reserved for direct POC evaluation and strategy comparison; its label is never passed into feature engineering or model fitting.

In [ ]:
assert TARGET in test_raw.columns, 'This shortened workflow requires labeled test data.'
y_test = test_raw[TARGET].astype(int)
test_amount = test_raw['transaction_amount'].to_numpy()

print(f'Full training set: {len(X_all):,} rows, {y_all.sum():,} fraud cases ({y_all.mean():.4%})')
print(f'Test set:          {len(X_test_all):,} rows, {y_test.sum():,} fraud cases ({y_test.mean():.4%})')

## Baseline, models, and capacity metrics

The rule baseline is intentionally transparent. Update its thresholds below only if your Phase 1 EDA established a different rule; keep the rule fixed before comparing it with ML.

In [ ]:
PHASE_1_HIGH_RISK_TYPES = {'TRANSFER', 'CASH_OUT'}
PHASE_1_HIGH_AMOUNT = 200_000

def phase_1_rule_score(df: pd.DataFrame) -> np.ndarray:
    """Explainable rule ranking: risky type, emptied sender balance, then transaction magnitude."""
    risky_type = df['transaction_type'].isin(PHASE_1_HIGH_RISK_TYPES).astype(float)
    sender_emptied = (df['sender_balance_after'] == 0).astype(float)
    high_amount = (df['transaction_amount'] >= PHASE_1_HIGH_AMOUNT).astype(float)
    # The small amount term breaks ties without overriding the rule bands.
    amount_tiebreak = np.log1p(df['transaction_amount'].clip(lower=0)) / 1_000
    return (2 * risky_type + sender_emptied + high_amount + amount_tiebreak).to_numpy()

def signed_log1p(values):
    values = np.asarray(values, dtype=float)
    return np.sign(values) * np.log1p(np.abs(values))

numeric_features = X_all.select_dtypes(include=np.number).columns.tolist()
categorical_features = ['transaction_type']

linear_preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('signed_log', FunctionTransformer(signed_log1p)),
            ('scale', StandardScaler()),
        ]), numeric_features),
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('one_hot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', SimpleImputer(strategy='median'), numeric_features),
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), categorical_features),
    ],
    sparse_threshold=0,
)

positive_count = y_all.sum()
negative_count = len(y_all) - positive_count
SCALE_POS_WEIGHT = negative_count / positive_count
print(f'Class-imbalance weight: {SCALE_POS_WEIGHT:.1f}')

models = {
    'Logistic regression': Pipeline([
        ('preprocess', linear_preprocessor),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1_000, solver='lbfgs', random_state=RANDOM_STATE)),
    ]),
    'XGBoost': Pipeline([
        ('preprocess', tree_preprocessor),
        ('model', XGBClassifier(
            objective='binary:logistic', eval_metric='aucpr', tree_method='hist',
            n_estimators=400, max_depth=5, learning_rate=0.05,
            subsample=0.80, colsample_bytree=0.80, min_child_weight=5,
            reg_lambda=5.0, scale_pos_weight=SCALE_POS_WEIGHT,
            random_state=RANDOM_STATE, n_jobs=-1,
        )),
    ]),
    'LightGBM': Pipeline([
        ('preprocess', tree_preprocessor),
        ('model', LGBMClassifier(
            objective='binary', n_estimators=400, learning_rate=0.05,
            num_leaves=31, max_depth=-1, subsample=0.80, colsample_bytree=0.80,
            reg_lambda=5.0, scale_pos_weight=SCALE_POS_WEIGHT,
            random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
        )),
    ]),
}

def capacity_metrics(y_true, scores, amounts, capacity=INVESTIGATION_CAPACITY):
    """Evaluate a ranking when investigators can review only the top capacity share."""
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    amounts = np.asarray(amounts, dtype=float)
    review_count = max(1, ceil(len(y_true) * capacity))
    ranked = np.argsort(-scores, kind='stable')
    reviewed = ranked[:review_count]
    fraud_found = y_true[reviewed].sum()
    total_fraud = y_true.sum()
    precision = fraud_found / review_count
    recall = fraud_found / total_fraud if total_fraud else np.nan
    value_found = amounts[reviewed][y_true[reviewed] == 1].sum()
    total_fraud_value = amounts[y_true == 1].sum()
    value_recall = value_found / total_fraud_value if total_fraud_value else np.nan
    return {
        'reviews': review_count,
        'fraud_cases_found': fraud_found,
        'recall_at_1pct': recall,
        'precision_at_1pct': precision,
        'lift_at_1pct': precision / y_true.mean() if y_true.mean() else np.nan,
        'fraud_value_recall_at_1pct': value_recall,
        'reviews_per_fraud': review_count / fraud_found if fraud_found else np.inf,
        'pr_auc': average_precision_score(y_true, scores),
        'roc_auc': roc_auc_score(y_true, scores),
    }

In [ ]:
test_results = []
fitted_candidates = {}

rule_scores = phase_1_rule_score(test_raw)
test_results.append({'model': 'Phase 1 rule', **capacity_metrics(y_test, rule_scores, test_amount)})

for model_name, model in models.items():
    print(f'Fitting {model_name}...')
    fitted = clone(model).fit(X_all, y_all)
    probabilities = fitted.predict_proba(X_test_all)[:, 1]
    fitted_candidates[model_name] = fitted
    test_results.append({'model': model_name, **capacity_metrics(y_test, probabilities, test_amount)})

test_comparison = (
    pd.DataFrame(test_results)
    .sort_values(['recall_at_1pct', 'precision_at_1pct'], ascending=False)
    .reset_index(drop=True)
)
display(test_comparison.style.format({
    'recall_at_1pct': '{:.2%}', 'precision_at_1pct': '{:.2%}', 'lift_at_1pct': '{:.1f}x',
    'fraud_value_recall_at_1pct': '{:.2%}', 'reviews_per_fraud': '{:.1f}',
    'pr_auc': '{:.4f}', 'roc_auc': '{:.4f}',
}))

ml_comparison = test_comparison.query("model != 'Phase 1 rule'").reset_index(drop=True)
selected_model_name = ml_comparison.loc[0, 'model']
baseline_wins = test_comparison.loc[0, 'model'] == 'Phase 1 rule'
print(f'Selected ML model by Recall@1% then Precision@1%: {selected_model_name}')
if baseline_wins:
    print('Decision gate: the Phase 1 rule currently outperforms ML at 1%; retain the rule unless other business evidence changes the decision.')

## Calibrate the selected model and compare queue strategies

Calibration makes the score more suitable for the exposure strategy, `P(fraud) × transaction amount`. It is fit with 3-fold calibration on the full training dataset; test labels remain unseen during calibration.

In [ ]:
selected_base_model = models[selected_model_name]
calibrated_model = CalibratedClassifierCV(
    estimator=clone(selected_base_model), method='sigmoid', cv=3, n_jobs=-1
).fit(X_all, y_all)

test_probability = calibrated_model.predict_proba(X_test_all)[:, 1]
test_exposure = test_probability * test_amount

strategy_results = pd.DataFrame([
    {'strategy': 'A — fraud probability', **capacity_metrics(y_test, test_probability, test_amount)},
    {'strategy': 'B — expected gross exposure', **capacity_metrics(y_test, test_exposure, test_amount)},
])
display(strategy_results.style.format({
    'recall_at_1pct': '{:.2%}', 'precision_at_1pct': '{:.2%}', 'lift_at_1pct': '{:.1f}x',
    'fraud_value_recall_at_1pct': '{:.2%}', 'reviews_per_fraud': '{:.1f}',
    'pr_auc': '{:.4f}', 'roc_auc': '{:.4f}',
}))

fig, ax = plt.subplots(figsize=(6, 4))
CalibrationDisplay.from_predictions(y_test, test_probability, n_bins=10, strategy='quantile', ax=ax)
ax.set_title('Selected model calibration — test set')
plt.show()

# Default policy: protect the most money only when it achieves a material value gain without a major loss in cases.
MIN_VALUE_RECALL_GAIN = 0.05
MIN_CASE_RECALL_RATIO = 0.90
probability_row = strategy_results.iloc[0]
exposure_row = strategy_results.iloc[1]
use_exposure = (
    exposure_row['fraud_value_recall_at_1pct'] - probability_row['fraud_value_recall_at_1pct'] >= MIN_VALUE_RECALL_GAIN
    and exposure_row['recall_at_1pct'] >= MIN_CASE_RECALL_RATIO * probability_row['recall_at_1pct']
)
selected_strategy = 'B — expected gross exposure' if use_exposure else 'A — fraud probability'
print(f'Recommended policy under the configured guardrail: {selected_strategy}')

## POC test result

The test label is dropped before scoring. Its metrics summarize the selected policy at the fixed 1% capacity. Because this shortened workflow selected the model and strategy using these test results, treat this as POC evidence rather than an independent production estimate.

In [ ]:
test_priority_score = test_probability * test_amount if use_exposure else test_probability

if TARGET in test_raw.columns:
    y_test = test_raw[TARGET].astype(int)
    final_test_results = pd.DataFrame([
        {'strategy': selected_strategy, **capacity_metrics(y_test, test_priority_score, test_amount)}
    ])
    display(final_test_results.style.format({
        'recall_at_1pct': '{:.2%}', 'precision_at_1pct': '{:.2%}', 'lift_at_1pct': '{:.1f}x',
        'fraud_value_recall_at_1pct': '{:.2%}', 'reviews_per_fraud': '{:.1f}',
        'pr_auc': '{:.4f}', 'roc_auc': '{:.4f}',
    }))

    print('Interpretation: these metrics estimate how much fraud and fraudulent value the fixed 1% queue captures.')
else:
    print('No test labels found: ranked recommendations will be exported without test performance metrics.')

## Export recommendations

The selected calibrated model has already been fit on all labeled training data. The export intentionally omits `is_fraud`, because an investigator would not know that label at decision time.

In [ ]:
final_probability = test_probability
final_priority_score = final_probability * test_amount if use_exposure else final_probability
review_count = max(1, ceil(len(test_raw) * INVESTIGATION_CAPACITY))
ranking = pd.Series(final_priority_score, index=test_raw.index).rank(method='first', ascending=False).astype(int)

scored_transactions = test_raw.drop(columns=[TARGET], errors='ignore').copy()
scored_transactions.insert(0, 'source_row', np.arange(len(scored_transactions)))
scored_transactions['fraud_risk_score'] = final_probability
scored_transactions['priority_score'] = final_priority_score
scored_transactions['investigation_rank'] = ranking
scored_transactions['recommend_investigation'] = ranking <= review_count

queue_columns = [
    'source_row', 'sender_id', 'recipient_id', 'transaction_type', 'transaction_amount',
    'fraud_risk_score', 'priority_score', 'investigation_rank', 'recommend_investigation',
]
queue_columns = [column for column in queue_columns if column in scored_transactions.columns]
investigation_queue = scored_transactions.loc[scored_transactions['recommend_investigation'], queue_columns]

scored_path = OUTPUT_DIR / 'test_scored_transactions.csv'
queue_path = OUTPUT_DIR / 'top_1pct_investigation_queue.csv'
scored_transactions.to_csv(scored_path, index=False)
investigation_queue.to_csv(queue_path, index=False)

print(f'Exported {len(scored_transactions):,} scored test transactions to: {scored_path.resolve()}')
print(f'Exported {len(investigation_queue):,} investigation recommendations to: {queue_path.resolve()}')
display(investigation_queue.head(10))

## Presentation takeaway

Use the final-test table to make a capacity-constrained statement: *At the same fixed 1% investigator capacity, the selected model captures [recall] of fraud cases, creates a queue with [precision] precision, and concentrates fraud [lift] times above the overall rate.*

Limitations to state: the data are synthetic, the observed rate is not a production prevalence estimate, balance-after fields must be available at scoring time, and the model needs monitoring and retraining with investigator outcomes in production.